# EchoCV behaviour detector → LiteRT (.tflite)

LiteRT export only runs on Linux x86_64 or macOS, so run this in Google Colab (CPU runtime is enough).

1. Run the cells in order. The upload cell asks for `ml/artifacts/yolo/v0/best.pt`.
2. Download `best_w8a32.tflite` and save it as `ml/artifacts/yolo/v0/best_w8a32.tflite`.

`w8a32` = dynamic INT8 weights, float activations; it needs no calibration data.

In [ ]:
!pip install -q "ultralytics>=8.4.83"

In [ ]:
from google.colab import files

uploaded = files.upload()  # choose ml/artifacts/yolo/v0/best.pt
weights = next(iter(uploaded))

In [ ]:
import shutil
from pathlib import Path

from ultralytics import YOLO

model = YOLO(weights)
out = Path(model.export(format="litert", quantize="w8a32", imgsz=640))
target = Path("best_w8a32.tflite")
shutil.copyfile(next(out.glob("*.tflite")) if out.is_dir() else out, target)
print(target, target.stat().st_size / 1e6, "MB")

In [ ]:
# Sanity check: the .tflite and .pt should find the same classes on a sample image.
import urllib.request

urllib.request.urlretrieve("https://ultralytics.com/images/zidane.jpg", "sample.jpg")
for w in (weights, str(target)):
    r = YOLO(w, task="detect")("sample.jpg", imgsz=640, conf=0.25, verbose=False)[0]
    boxes = zip(r.boxes.cls, r.boxes.conf, strict=True)
    print(w, [(r.names[int(c)], round(float(p), 2)) for c, p in boxes])

In [ ]:
files.download(str(target))